# MT Hexapod current noise at faults

Author: Diego Hurtado\
Last Update: 11/23/2025\
\
This notebook will search timestamps for M2Hex faults
and analyze the current noise when it faults.\
We will only consider Faults with error codes equal to 1\
For a robust analysis, this notebook anaylzes since the start of observations in April

Noise being the standard deviation of the current in the motors, given by the EfdClient

M2Hex has several faults associated with motor oscillations. We want to characterize these oscillations/vibrations.\
I suggest the creation of a function that applies FFT to a motor current and extracts the most dominant frequency components.\
I will leave the technical aspects of the implementation open for now since there might be some pre-processing we need to apply to the data.

In [ ]:
import logging
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sb
from tqdm.notebook import tqdm

from astropy.time import Time, TimeDelta
from datetime import datetime, timedelta # For days_between function
from scipy.fft import fft, ifft
from scipy.signal import detrend, find_peaks


import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsStartTime,
    makeEfdClient,
)

# Create the client for InfluxQL
efd_client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=False)

logger = logging.getLogger("M2Hex_frequencies")

In [ ]:
logger.setLevel("INFO")

In [ ]:
# Output directory
directory = "2190_plots"
notebook_dir = os.getcwd()
directory = os.path.join(notebook_dir, f"{directory}")
os.makedirs(directory, exist_ok=True)
print("Folder created at:", directory)

error_code = 1  # Will only consider errorCode 1 as Faults
sal_index = 2  # M2Hexapods (maybe make it to search for CamHex faults aswell?)
n_struts = 6 # Amount of Hex Struts for when we look at all of them

# Upper and lower time difference from the fault
delta_start = 10
delta_end = 0

# EFD Topics configuration

columns = {
    "lsst.sal.MTHexapod.logevent_errorCode": ["MTHexapodID", "errorCode", "errorReport", "salIndex"],
    "lsst.sal.MTMount.elevation": ["actualPosition"],
    "lsst.sal.MTMount.azimuth": ["actualPosition"],

}

# Pandas configuration
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

To simplify functions lets make them search for the dataframe\
with a start and an end. Then the same function searches for\
the useful plot and the context plot (is this really necessary?).\
The idea is forking the functions


In [ ]:
# Stablish functions

def timestamp_deltas(timestamp, delta_start, delta_end):
    """
    Creates the timestamps to query errorCodes

    Parameters
    ----------
    timestamp: string (YYYY-MM-DD hh:mm:ss.ms)
        Time when error occurred
    delta_start: int
        Delta in seconds before error
    delta_end: int
        Delta in seconds after error
    
    Returns:
    ----------
    EFD-Readable timestamps
    """
    
    timestamp = timestamp_to_utc(timestamp)
    
    delta1 = TimeDelta(delta_start, format="sec")
    delta2 = TimeDelta(delta_end, format="sec")

    start_time = (timestamp - delta1).to_value("isot", subfmt="date_hms") + "Z"
    end_time = (timestamp + delta2).to_value("isot", subfmt="date_hms") + "Z"

    start_plot = (timestamp - TimeDelta("60", format="sec")).to_value(
        "isot", subfmt="date_hms"
    ) + "Z"
    end_plot = (timestamp + TimeDelta("5", format="sec")).to_value(
        "isot", subfmt="date_hms"
    ) + "Z"

    return start_time, end_time, start_plot, end_plot


def days_between(start, end):
    """
    Creates a list of YYYYMMDD days for analysis
    
    Parameters:
    -----------
    start : int (YYYYMMDD)
        Start date
    end : int (YYYYMMDD)
        End date
    
    Returns:
    --------
    List with YYYYMMDD in between dates given
    """
    start_date = datetime.strptime(str(start), "%Y%m%d")
    end_date = datetime.strptime(str(end), "%Y%m%d")
    
    days = []
    current = start_date
    while current <= end_date:
        days.append(int(current.strftime("%Y%m%d")))
        current += timedelta(days=1)
    
    return days


def timestamp_to_utc(timestamp):
    """
    Takes the timestamp in string value and
    converts it to a query readable timestamp
    
    Parameter
    ---------
    Timestamp: string (YYYY-MM-DD hh:mm:ss.ms)

    Returns:
    --------
    EFD-readable timestamp
    """
    timestamp = pd.to_datetime(timestamp)
    new_stamp = timestamp.value
    step = int(0.01 * 1e10)
    rounded_stamp = (new_stamp // step) * step
    timestamp = Time(pd.to_datetime(rounded_stamp), scale="utc")

    return timestamp


def current_query(motor, start_time, end_time):
    """
    Query the noise current for a certain motor in a timelapse given by the deltas and timestamp
    Returns the Motor Current dataframe

    Parameters
    ----------
    motor: integer
        Strut ID for the motorCurrent to query
    start_time:

    end_time:
        
    """
    logger.debug(f"Querying for Current{motor}")
    
    start_time = timestamp_to_utc(start_time)
    end_time = timestamp_to_utc(end_time)
    
    df_current = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.electrical",
        columns=[f"motorCurrent{motor}"],
        begin=start_time,
        end=end_time,
        )
    
    if df_current is None or len(df_current) == 0:
        print(f"Failed query for Current of motor {motor} @ {start_time}")
        empty_df = df_current.copy() if df_current is not None else pd.DataFrame()
        empty_df[f"Current {motor}"] = []

        return empty_df
    
    return df_current


def alt_az_query(start_time, end_time):
    """
    Query the altitude and azimuth for a given timestamp window
    Returns elevation and azimuths for the error window and context window

    Parameters
    ----------
    delta_start: integer
        Time in seconds before the timestamp
    delta_end: integer
        Time in seconds after the timestamp
    timestamp: string (YYYY-MM-DD hh:mm:ss.ms)
        Time when error occurred
    """
    
    start_time = timestamp_to_utc(start_time)
    end_time = timestamp_to_utc(end_time)

    logger.debug("Querying for Elevation")
    df_alt = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.elevation",
        columns=["actualPosition"],
        begin=start_time,
        end=end_time,
        )

    logger.debug("Querying for Azimuth")
    df_az = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTMount.azimuth",
        columns=["actualPosition"],
        begin=start_time,
        end=end_time,
        )

    df_alt.index = pd.to_datetime(df_alt.index)
    df_az.index  = pd.to_datetime(df_az.index)
    
    df_alt_az = pd.merge_asof(
        df_alt.sort_index(), df_az.sort_index(),
        left_index=True, right_index=True,
        tolerance=pd.Timedelta('100ms')
    )
    logger.debug(df_alt_az.index)

    if df_alt_az is None or len(df_alt_az) == 0:
        print(f"Failed query for Elevation and Azimuth @ {start_time}")
        empty_df = df_alt_az.copy() if df_alt_az is not None else pd.DataFrame()
        empty_df['Elevation'] = [0]
        empty_df['Azimuth'] = [0]

        return empty_df
        
    df_alt_az.columns = ['Elevation', 'Azimuth']
    
    return df_alt_az


def strut_pos_query(motor, start_time, end_time):
    """
    Queries the Calibrated position of the Struts (motor)


    Parameters
    ----------
    """
    logger.debug(f"Querying for Position {motor}")
    
    start_time = timestamp_to_utc(start_time)
    end_time = timestamp_to_utc(end_time)
    
    df_position= getEfdData(
        client=efd_client,
        topic="lsst.sal.MTHexapod.actuators",
        columns=[f"calibrated{motor}"],
        begin=start_time,
        end=end_time,
        )

    df_position.index = pd.to_datetime(df_position.index)
    df_position = df_position.resample('100ms').mean()
    
    if df_position is None or len(df_position) == 0:
        print(f"Failed query for Position of motor {motor} @ {start_time}")
        empty_df = df_position.copy() if df_position is not None else pd.DataFrame()
        empty_df[f"Position motor {motor}"] = []

        return empty_df
    
    df_position.columns = [f"Position motor {motor}"]
    
    return df_position


def fft_positive_values(motor, df, dt):
    """
    Computes and returns in dataframes the Positive values
    of a shifted FFT, of its frequencies and peaks indexes
    and of the detrended signal.

    Parameters
    ----------
    """
    logger.debug(f"Calculating FFT values motor{motor}")
    
    signal = df.values.flatten()
    signal = detrend(signal)
    time = df.index

    fft_vals[motor] = np.fft.fftshift(np.fft.fft(signal))
    real_fft_vals[motor] = np.abs(np.real(fft_vals[motor]))
    imag_fft_vals[motor] = np.imag(fft_vals[motor])
    freqs[motor] = np.fft.fftshift(np.fft.fftfreq(len(signal), dt))
    mask = freqs[motor] > 0
    freqs[motor] = freqs[motor][mask]
    real_fft_vals[motor] = real_fft_vals[motor][mask]
    imag_fft_vals[motor] = imag_fft_vals[motor][mask]

    threshold = 0.5 * np.max(np.abs(real_fft_vals[motor]))
    peaks_index[motor], properties[motor] = find_peaks(
        real_fft_vals[motor], height=threshold
    )  # Modify height for distance

    df_signal[motor] = pd.DataFrame({"Time": time, f"Detrend signal": signal})
    df_signal[motor]["Time"] = pd.to_datetime(df_signal[motor]["Time"])
    df_signal[motor].set_index("Time", inplace=True)

    return (real_fft_vals[motor],
            imag_fft_vals[motor],
            freqs[motor],
            peaks_index[motor],
            df_signal[motor])

## Frequencies when Faulting
We are going to look for errorcodes in said day interval\
and analyze for the currents in them.

In [ ]:
# Days to study
day_start = 20240501
day_end = 20251120

daysinbetween = days_between(day_start, day_end)
print(f"We have {len(daysinbetween)} days to analyze")

start_time = getDayObsStartTime(day_start)
end_time = getDayObsEndTime(day_end)
error_key = "lsst.sal.MTHexapod.logevent_errorCode"

# Query Hexapod faults
df_timestamps = getEfdData(
    client=efd_client,
    topic= error_key,
    columns=columns[error_key],
    begin=start_time,
    end=end_time,
)

df_timestamps = df_timestamps[df_timestamps.salIndex == sal_index]

# Know which Timestamps caused errorCode = 1 Faults
timestamps_error_code = df_timestamps[df_timestamps.errorCode == error_code].index
print(f"and {len(timestamps_error_code)} faults to check")
logger.debug(timestamps_error_code)

In [ ]:
fig, ax = plt.subplots(3, 1, dpi=128, figsize=(21, 14))
signal_threshold = 0.75

median_val = {}
df_current = {}
median_df = {}

for motor in range(n_struts):
    median_val[motor] = []

timestamp_list = []
dt_list = []

# Cycle through timestamps and motors, then append each to their respective list
for timestamp in timestamps_error_code:
    timestamp_list.append(timestamp)

    for motor in range(n_struts):
        start_time, end_time, _,_ = timestamp_deltas(timestamp, delta_start, delta_end)
        df_current[motor] = current_query(motor, start_time, end_time)
        noisemedian = df_current[motor][f"motorCurrent{motor}"].median()
        median_val[motor].append(noisemedian)
        dt = df_current[motor].index.diff().median().total_seconds()
        dt_list.append(dt)

# With the above ready, we cycle again through motors

for motor in range(n_struts):
    df = pd.DataFrame(
        {"Time": timestamp_list, f"Median Noise {motor}": median_val[motor]}
    )

    df["Time"] = pd.to_datetime(df["Time"])
    df.set_index("Time", inplace=True)
    median_df[motor] = df.dropna()

    plot_df = median_df[motor][median_df[motor][f"Median Noise {motor}"] > signal_threshold].index
    bins = round((len(median_df[motor].index)) / 20)
    ax[0].hist(plot_df, bins=bins, alpha=0.5, label=f"Motor {motor}")

ax[0].set_xlabel(f"Time (Bins={bins})")
ax[0].set_ylabel("Frequency")
ax[0].tick_params(axis='x', labelrotation=20)
ax[0].set_title(f"Times when median noise > {signal_threshold}")
ax[0].legend()
ax[0].grid(True)

median_df[5].plot(ax=ax[1], alpha=0.5, color="C1")
median_df[4].plot(ax=ax[1], alpha=0.5, color="C0")
median_df[3].plot(ax=ax[1], alpha=0.5, color="C4")
# median_df[2].plot(ax=ax[1], alpha=0.5, color='C7')
# median_df[1].plot(ax=ax[1], alpha=0.5, color='C8')
# median_df[0].plot(ax=ax[1], alpha=0.5, color='C9')
ax[1].set_ylabel("Median of Noise current [A]")
ax[1].set_xlabel("Error timestamps")
ax[1].grid(True)

bins = int(len(dt_list) * 0.025)
dt_mean = np.mean(dt_list)
dt_median = np.median(dt_list)
dt_sigma = np.std(dt_list)

ax[2].hist(dt_list, bins=bins, color="C1")
ax[2].set_xlabel("Sampling frequencies")
ax[2].set_ylabel("Ocassions")
ax[2].set_title("Histogram of sampling frequencies")
ax[2].grid(True)

filename = "sampling_and_current.png"
path = os.path.join(directory, filename)
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.tight_layout()
plt.show()

print(f" Mean: {round(dt_mean, 4)}s \n Median: {round(dt_median, 4)}s\n Sigma: {round(dt_sigma, 4)}s")

# Delete values for RAM efficiency
del median_df[0],median_df[1],median_df[2],median_df[3],median_df[4],median_df[5]
del df_current[0],df_current[1],df_current[2],df_current[3],df_current[4],df_current[5]

As we can notice, the sampling frequency is not the same across all data,\
being the Median equal to roughly 40 Hz.\
For the Fourier Transform in a discrete set of data,\
this means that we can only trust results in frequencies less than half of it,\
This correspond to the Nyquist frequency, in this case ~ 20 Hz.\
\
For robustness of statistics and across all data, we may choose the median for the FFTs.

In [ ]:
dt = dt_median

## Oscillations of the current

To understand how the current behaves we have to see the components of the frequency.\
\
The next part of the code also gathers the dominant frequencies for both Struts 5 and 6.\
It attaches them to the same 1st Dominant and 2nd Dominant Frequency lists\
without discriminating to which strut it corresponds to, \
but considering both Struts exert similar frequencies for each instance, \
this can be overlooked to gather a Table with same-sized lists
### The next part of the code takes ~10 minutes to run

In [ ]:
# First let's create a folder to store these plots
output_dir = f"{directory}/m2hex_plots"
os.makedirs(output_dir, exist_ok=True)

df_current = {}
df_position = {}
peaks_index = {}
first_peak = {}
second_peak = {}
properties = {}
df_signal = {}
fft_vals = {}
real_fft_vals = {}
imag_fft_vals = {}
freqs = {}
power = {}
motor_pos = {}

timestamp_list = []
altitude_list = []
first_peak[4] = []
second_peak[4] = []
first_peak[5] = []
second_peak[5] = []
motor_pos[4] = []
motor_pos[5] = []

for timestamp in tqdm(timestamps_error_code):
    
    #print(f"Time:{timestamp}")
    
    start_time, end_time, start_plot, end_plot = timestamp_deltas(timestamp, delta_start, delta_end)
    df_alt_az = alt_az_query(start_time, end_time)
    altmedian = df_alt_az[f"Elevation"].median()
    logger.debug(altmedian)

    for motor in range(3, 6):
        
        df_current[motor] = current_query(motor, start_time, end_time)
        df_position[motor] = strut_pos_query(motor, start_time, end_time)
        
        logger.debug(df_current[motor])
        
        logger.debug(type(df_current[motor]))
        
        (real_fft_vals[motor],
            imag_fft_vals[motor],
            freqs[motor],
            peaks_index[motor],
            df_signal[motor]) = fft_positive_values(motor, df_current[motor], dt)
    
    if len(peaks_index[4]) < 2 or len(peaks_index[5]) < 2:
        logger.debug(f"Sizes {len(peaks_index[4])} and {len(peaks_index[5])} don't suffice")
        continue

    altitude_list.append(altmedian)
    timestamp_list.append(timestamp)

    min_hz_distance = 0.0  # This should not be 0

    for motor in range(4, 6):
        chosen_peaks = []
        chosen_peaks.append(int(peaks_index[motor][-1]))
        for peak in peaks_index[motor][:-1]:
            if (freqs[motor][chosen_peaks] - freqs[motor][peak] > min_hz_distance):  
                # This is always true, how do I make sure I'm doing this right?
                chosen_peaks.append(int(peak))
            if len(chosen_peaks) == 2:
                f1 = freqs[motor][chosen_peaks[0]]
                f2 = freqs[motor][chosen_peaks[1]]
                first_peak[motor].append(f1)
                second_peak[motor].append(f2)

                position_median = df_position[motor][f"Position motor {motor}"].median()
                motor_pos[motor].append(position_median)
                logger.debug(position_median)

                break
    
    '''
    Apparently the code does not always gather the Position of
    Struts or Telescope, so lets handle that while we search for a solution
    '''
    
    if not df_position[4].empty:
        fig, ax = plt.subplots(4, 1, dpi=128, figsize=(20, 30))
    else:
        fig, ax = plt.subplots(3, 1, dpi=128, figsize=(20, 20))
    
    df_signal[4].plot(ax=ax[0], color="C0")
    df_signal[5].plot(ax=ax[0], color="C1")
    df_signal[3].plot(ax=ax[0], color="C4")
    ax[0].set_ylabel("Current [A]")
    ax[0].set_xlabel("Time")
    ax[0].set_title(f"Raw Current for analisys")
    ax[0].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
    ax[0].axvline(x=start_time, color="C2", linestyle="-")
    ax[0].grid(True)
    ax[0].legend(handles=None)

    ax[1].plot(freqs[4], real_fft_vals[4], color="C0", label=f"Real Motor 4")
    ax[1].scatter(
        freqs[4][peaks_index[4]],
        properties[4]["peak_heights"],
        marker="x",
        s=100,
        color="C4",
        label="Peaks Motor 4")
    ax[1].plot(freqs[5], real_fft_vals[5], color="C1", label=f"Real Motor 5")
    ax[1].scatter(
        freqs[5][peaks_index[5]],
        properties[5]["peak_heights"],
        marker="x",
        s=100,
        color="C3",
        label="Peaks Motor 5")
    ax[1].plot(freqs[4], imag_fft_vals[4], color="C0", alpha=0.25, label=f"Imaginary Motor 4")
    ax[1].plot(freqs[5], imag_fft_vals[5], color="C1", alpha=0.25, label=f"Imaginary Motor 5")
    ax[1].set_xlabel("Frequency [Hz]")
    ax[1].set_ylabel("Power")
    ax[1].axvline(
        x=(1 / dt) / 2,
        color="C3",
        linestyle="-",
        label=f"Nyquist frequency = {(1 / dt) / 2}")
    _, right = ax[1].get_xlim()
    ax[1].set_xlim(0,right)
    ax[1].set_title(f"Fast Fourier Transforms")
    ax[1].legend()
    ax[1].grid(True)
    
    df_alt_az["Elevation"].plot(ax=ax[2], color="C1")
    ax[2].set_xlabel("Time")
    ax[2].set_ylabel("Degrees elevation")
    ax[2].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
    ax[2].axvline(x=start_time, color="C2", linestyle="-")
    ax[2].set_title(f"Elevation of telescope")
    ax[2].legend()
    ax[2].grid(True)
    
    if not df_position[4].empty:
        df_position[5].plot(ax=ax[3], color="C1")
        df_position[3].plot(ax=ax[3], color="C4")
        df_position[4].plot(ax=ax[3], color="C0")
        ax[3].set_xlabel("Time")
        ax[3].set_ylabel("Position in [nm]")
        ax[3].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
        ax[3].axvline(x=start_time, color="C2", linestyle="-")
        ax[3].set_title(f"Calibrated position of struts")
        ax[3].legend()
        ax[3].grid(True)
    
    filename = f"{timestamp}.png"
    path = os.path.join(output_dir, filename)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    #plt.show()
    plt.close()

In [ ]:
logger.debug(
    f"Make sure these lengths are the same:{
        (
            len(timestamp_list),
            len(altitude_list),
            len(motor_pos[4]),
            len(motor_pos[5]),
            len(first_peak[4]),
            len(second_peak[4]),
            len(first_peak[5]),
            len(second_peak[5]),
        )
    }"
)

In [ ]:
df = pd.DataFrame(
    {
        "Time": timestamp_list,
        "Elevation": altitude_list,
        "Position Strut 5": motor_pos[4],
        "Position Strut 6": motor_pos[5],
        "1st Frequency Strut 5": first_peak[4],
        "2nd Frequency Strut 5": second_peak[4],
        "1st Frequency Strut 6": first_peak[5],
        "2nd Frequency Strut 6": second_peak[5],
    }
)
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time", inplace=True)
elevation_frequencies = df
matrix = elevation_frequencies.corr()

fig, ax = plt.subplots(3, 1, figsize=(10, 20))

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 7)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend="both")

sb.heatmap(matrix, annot=True, cmap=cmap, center=0.2, norm=norm, ax=ax[0])
ax[0].set_title("Correlation Matrix")
ax[0].tick_params(axis="x", rotation=45)


df["1st Frequency Strut 5"].hist(ax=ax[1], bins=20, color="C0", alpha=0.7, label="Strut 5")
df["2nd Frequency Strut 5"].hist(ax=ax[1], bins=20, color="C0", alpha=0.7)
df["1st Frequency Strut 6"].hist(ax=ax[1], bins=20, color="C1", alpha=0.7)
df["2nd Frequency Strut 6"].hist(ax=ax[1], bins=20, color="C1", alpha=0.7, label="Strut 6")
ax[1].set_xlabel("Frequencies")
ax[1].set_ylabel("Ocurrences")
ax[1].set_title(f"Histogram of dominant frequencies on {len(timestamp_list)} occasions")
ax[1].legend()
ax[1].grid(True)


df["Position Strut 5"].hist(ax=ax[2], color="C0", alpha=0.7, label="Strut 5")
df["Position Strut 6"].hist(ax=ax[2], color="C1", alpha=0.7, label="Strut 6")
ax[2].set_xlabel("Position [nm]")
ax[2].set_ylabel("Ocurrences")
ax[2].set_title(f"Histogram of positions on {len(timestamp_list)} occasions")
ax[2].legend()
ax[2].grid(True)

plt.tight_layout()

filename = "fault_summarize.png"
path = os.path.join(directory, filename)
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Telescope elevation vs Dominant Frequencies

fig, ax = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

ax[0].scatter(df["1st Frequency Strut 5"], df["Elevation"], color="C0", label="1st Dominant")
# ax[0].scatter(df["2nd Frequency Strut 5"], df["Elevation"], color='C1', label="2nd Dominant")
ax[0].set_ylabel("Telescope Elevation")
ax[0].set_xlabel("Frequency")
ax[0].set_title("Strut 5: Telescope elevation vs Frequencies")
ax[0].set_xlim(17.5, 20.5)
ax[0].legend()
ax[0].grid(True)

ax[1].scatter(df["1st Frequency Strut 6"], df["Elevation"], color="C1", label="1st Dominant")
# ax[1].scatter(df["2nd Frequency Strut 6"], df["Elevation"], color='C1', label="2nd Dominant")
ax[1].set_xlabel("Frequencies")
ax[1].set_title("Strut 6: Telescope elevation vs Frequencies")
ax[1].set_xlim(17.5, 20.5)
ax[1].legend()
ax[1].grid(True)

plt.tight_layout()


filename = "fault_elev_vs_freqs.png"
path = os.path.join(directory, filename)
plt.savefig(path, dpi=200, bbox_inches="tight")

plt.show()

## Frequencies when exposing
\
Along side the analysis when the Struts go to fault\
I want to see how this affects the exposures.\
Do these (or which) frequencies happen when exposing?\
I'll use the Butler for Exposure Starts and Ends.

In [ ]:
#day_start = 20240101 # ComCam
day_start = 20251120
day_end = 20251122

daysinbetween = days_between(day_start, day_end)
print(f"We have {len(daysinbetween)} days to analyze")

exp_start = []
exp_end = []

for day in daysinbetween: # This is a list
    logger.debug("Querying day", day)
    try:
        where = f"exposure.day_obs={day} AND instrument='LSSTCam'"
        records = butler.query_dimension_records('exposure', where=where, order_by='exposure.timespan.end')
        for r in records:
            exp_start.append(r.timespan.begin.isot)
            exp_end.append(r.timespan.end.isot)
    except:
        logger.info(f"Could not query day {day}")

df_exposures = pd.DataFrame({"Start": exp_start,
                               "End": exp_end
                            })

df_exposures.index.name = "Exposure"
print(f"Total exposures: {len(df_exposures)}")

I subestimated the amount of exposures\
\
Because of the amount of exposures to analyze,\
this next cell may take more than 15 minutes to run.

In [ ]:
output_dir = f"{directory}/exposure_plots"
os.makedirs(output_dir, exist_ok=True)

df_current = {}
df_position = {}
peaks_index = {}
first_peak = {}
second_peak = {}
properties = {}
df_signal = {}
fft_vals = {}
real_fft_vals = {}
imag_fft_vals = {}
freqs = {}
power = {}
motor_pos = {}

timestamp_list = []
altitude_list = []
first_peak[4] = []
second_peak[4] = []
first_peak[5] = []
second_peak[5] = []
motor_pos[4] = []
motor_pos[5] = []

dt = dt_median

for i in tqdm(range(len(df_exposures))):
    
    start_time = df_exposures["Start"][i]
    end_time = df_exposures["End"][i]
    
    df_alt_az = alt_az_query(start_time, end_time)
    altmedian = df_alt_az[f"Elevation"].median()
    logger.debug(altmedian)

    for motor in range(3, 6):
        
        df_current[motor] = current_query(motor, start_time, end_time)
        df_position[motor] = strut_pos_query(motor, start_time, end_time)
        
        logger.debug(df_current[motor])
        logger.debug(type(df_current[motor]))
        
        if df_current[motor] is None or len(df_current[motor]) == 0 or df_current[motor].empty:
            print("No current to analyze")
            continue
        else:
            (real_fft_vals[motor],
                imag_fft_vals[motor],
                freqs[motor],
                peaks_index[motor],
                df_signal[motor]) = fft_positive_values(motor, df_current[motor], dt)
    
    if len(peaks_index[4]) < 2 or len(peaks_index[5]) < 2:
        logger.debug(f"Sizes {len(peaks_index[4])} and {len(peaks_index[5])} don't suffice")
        continue

    altitude_list.append(altmedian)
    timestamp_list.append(start_time)

    min_hz_distance = 0.0  # This should not be 0

    for motor in range(4, 6):
        chosen_peaks = []
        chosen_peaks.append(int(peaks_index[motor][-1]))
        for peak in peaks_index[motor][:-1]:
            if (freqs[motor][chosen_peaks] - freqs[motor][peak] > min_hz_distance):  
                # This is always true, how do I make sure I'm doing this right?
                chosen_peaks.append(int(peak))
            if len(chosen_peaks) == 2:
                f1 = freqs[motor][chosen_peaks[0]]
                f2 = freqs[motor][chosen_peaks[1]]
                first_peak[motor].append(f1)
                second_peak[motor].append(f2)

                position_median = df_position[motor][f"Position motor {motor}"].median()
                motor_pos[motor].append(position_median)
                logger.debug(position_median)

                break
    
    if not df_position[4].empty:
        fig, ax = plt.subplots(4, 1, dpi=128, figsize=(20, 30))
    else:
        fig, ax = plt.subplots(3, 1, dpi=128, figsize=(20, 20))
    
    df_signal[4].plot(ax=ax[0], color="C0")
    df_signal[5].plot(ax=ax[0], color="C1")
    df_signal[3].plot(ax=ax[0], color="C4")
    ax[0].set_ylabel("Current [A]")
    ax[0].set_xlabel("Time")
    ax[0].set_title(f"Raw Current for analisys")
    ax[0].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
    ax[0].axvline(x=start_time, color="C2", linestyle="-")
    ax[0].grid(True)
    ax[0].legend(handles=None)

    ax[1].plot(freqs[4], real_fft_vals[4], color="C0", label=f"Real Motor 4")
    ax[1].scatter(
        freqs[4][peaks_index[4]],
        properties[4]["peak_heights"],
        marker="x",
        s=100,
        color="C4",
        label="Peaks Motor 4")
    ax[1].plot(freqs[5], real_fft_vals[5], color="C1", label=f"Real Motor 5")
    ax[1].scatter(
        freqs[5][peaks_index[5]],
        properties[5]["peak_heights"],
        marker="x",
        s=100,
        color="C3",
        label="Peaks Motor 5")
    ax[1].plot(freqs[4], imag_fft_vals[4], color="C0", alpha=0.25, label=f"Imaginary Motor 4")
    ax[1].plot(freqs[5], imag_fft_vals[5], color="C1", alpha=0.25, label=f"Imaginary Motor 5")
    ax[1].set_xlabel("Frequency [Hz]")
    ax[1].set_ylabel("Power")
    ax[1].axvline(
        x=(1 / dt) / 2,
        color="C3",
        linestyle="-",
        label=f"Nyquist frequency = {(1 / dt) / 2}")
    _, right = ax[1].get_xlim()
    ax[1].set_xlim(0,right)
    ax[1].set_title(f"Fast Fourier Transforms")
    ax[1].legend()
    ax[1].grid(True)
    
    df_alt_az["Elevation"].plot(ax=ax[2], color="C1")
    ax[2].set_xlabel("Time")
    ax[2].set_ylabel("Degrees elevation")
    ax[2].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
    ax[2].axvline(x=start_time, color="C2", linestyle="-")
    ax[2].set_title(f"Elevation of telescope")
    ax[2].legend()
    ax[2].grid(True)
    
    if not df_position[4].empty:
        df_position[5].plot(ax=ax[3], color="C1")
        df_position[3].plot(ax=ax[3], color="C4")
        df_position[4].plot(ax=ax[3], color="C0")
        ax[3].set_xlabel("Time")
        ax[3].set_ylabel("Position in [nm]")
        ax[3].axvline(x=end_time, color="C3", linestyle="-", label="Error Timestamp")
        ax[3].axvline(x=start_time, color="C2", linestyle="-")
        ax[3].set_title(f"Calibrated position of struts")
        ax[3].legend()
        ax[3].grid(True)
    
    filename = f"{start_time}.png"
    path = os.path.join(output_dir, filename)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    #plt.show()
    plt.close()
    
df = pd.DataFrame(
    {
        "Time": timestamp_list,
        "Elevation": altitude_list,
        "Position Strut 5": motor_pos[4],
        "Position Strut 6": motor_pos[5],
        "1st Frequency Strut 5": first_peak[4],
        "2nd Frequency Strut 5": second_peak[4],
        "1st Frequency Strut 6": first_peak[5],
        "2nd Frequency Strut 6": second_peak[5],
    }
)
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time", inplace=True)
elevation_frequencies = df
matrix = elevation_frequencies.corr()

plt.clf()
fig, ax = plt.subplots(3, 1, figsize=(12, 18))

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 9)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend="both")

sb.heatmap(matrix, annot=True, cmap=cmap, center=0.2, norm=norm, ax=ax[0])
ax[0].set_title("Correlation Matrix")
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45)


df["1st Frequency Strut 5"].hist(ax=ax[1], bins=20, color="C0", alpha=0.7, label="Strut 5")
df["2nd Frequency Strut 5"].hist(ax=ax[1], bins=20, color="C0", alpha=0.7)
df["1st Frequency Strut 6"].hist(ax=ax[1], bins=20, color="C1", alpha=0.7, label="Strut 6")
df["2nd Frequency Strut 6"].hist(ax=ax[1], bins=20, color="C1", alpha=0.7)
ax[1].set_xlabel("Frequencies")
ax[1].set_ylabel("Occurrences")
ax[1].set_title(f"Histogram of dominant frequencies on {len(timestamp_list)} occasions")
ax[1].legend()
ax[1].grid(True)

df["Position Strut 5"].hist(ax=ax[2], color="C0", alpha=0.7, label="Strut 5")
df["Position Strut 6"].hist(ax=ax[2], color="C1", alpha=0.7, label="Strut 6")
ax[2].set_xlabel("Position [nm]")
ax[2].set_ylabel("Occurrences")
ax[2].set_title(f"Histogram of positions on {len(timestamp_list)} occasions")
ax[2].legend()
ax[2].grid(True)

plt.tight_layout()

filename = "exposure_summarize.png"
path = os.path.join(directory, filename)
plt.savefig(path, dpi=200, bbox_inches='tight')
plt.show()

Few things to consider:
- The 0.0 Hz distance, it should be greater than that
- What happens if it doesn't have dominant frequencies? which would there be then?
- The result plots are about half the exposures (same as Faults)
- Similar results to Faults, only with anti-corr in Strut Position vs Elevation (expectable)
- What else can we do? This may not seem conclusive
- We have more frequencies in the 10 Hz mark